# HCV Workbench Reviewer Notebook

This notebook re-runs the reviewer-safe parts of the HTGAA HCV computational workbench from bundled files only. Regenerated artifacts are written to `notebook-output/` so the submitted `data/`, `figures/`, and `constructs/` files remain unchanged.

## 1. Setup

In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd() / "final-submission", Path("/content/final-submission")]
PACKAGE_ROOT = next((p.resolve() for p in candidates if (p / "data" / "sequence-manifest.tsv").exists()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Could not find final-submission/data/sequence-manifest.tsv. Upload or clone the final-submission folder first.")

os.chdir(PACKAGE_ROOT)
OUTPUT_ROOT = PACKAGE_ROOT / "notebook-output"
OUTPUT_ROOT.mkdir(exist_ok=True)

print(f"Package root: {PACKAGE_ROOT}")
print(f"Output root:  {OUTPUT_ROOT}")
print(f"Python:       {sys.version.split()[0]}")

In [ ]:
requirements = PACKAGE_ROOT / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
print("Setup complete")

## 2. Dataset Manifest Preview

In [ ]:
manifest_path = PACKAGE_ROOT / "data" / "sequence-manifest.tsv"
with manifest_path.open(newline="") as handle:
    manifest_rows = list(csv.DictReader(handle, delimiter="\t"))

print(f"Manifest rows: {len(manifest_rows)}")
print(f"Columns: {', '.join(manifest_rows[0].keys())}")
for row in manifest_rows[:5]:
    print({key: row[key] for key in ["accession", "genotype", "subtype", "include", "length_aa"]})

## 3. Run Core Reviewer Pipeline

In [ ]:
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir()

commands = [
    [sys.executable, "scripts/summarize_manifest.py"],
    [sys.executable, "scripts/run_basic_workbench.py"],
    [sys.executable, "scripts/export_construct.py", "--include-kozak"],
    [sys.executable, "scripts/draft_scorecard.py"],
    [sys.executable, "scripts/generate_plots.py"],
]
for command in commands:
    print("$", " ".join(command))
    completed = subprocess.run(command, text=True, capture_output=True, check=True)
    print(completed.stdout)

print("Generated files:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(PACKAGE_ROOT))

## 4. Validation Checks

In [ ]:
def read_fasta_sequence(path):
    return "".join(line.strip() for line in Path(path).read_text().splitlines() if line.strip() and not line.startswith(">"))

required_files = [
    OUTPUT_ROOT / "data" / "sequence-manifest-summary.tsv",
    OUTPUT_ROOT / "data" / "e2-conservation.csv",
    OUTPUT_ROOT / "data" / "e2-conserved-windows.csv",
    OUTPUT_ROOT / "data" / "design-scorecard.tsv",
    OUTPUT_ROOT / "figures" / "e2-conservation.svg",
    OUTPUT_ROOT / "figures" / "conservation-plot.svg",
    OUTPUT_ROOT / "constructs" / "final-candidate.protein.fasta",
    OUTPUT_ROOT / "constructs" / "final-candidate.dna.fasta",
]
missing = [str(path.relative_to(PACKAGE_ROOT)) for path in required_files if not path.exists()]
assert not missing, f"Missing expected outputs: {missing}"

with (OUTPUT_ROOT / "data" / "e2-conservation.csv").open(newline="") as handle:
    conservation_rows = list(csv.DictReader(handle))
assert conservation_rows, "No conservation rows generated"
assert all(0.0 <= float(row["conservation"]) <= 1.0 for row in conservation_rows), "Conservation scores outside 0-1"

with (OUTPUT_ROOT / "data" / "design-scorecard.tsv").open(newline="") as handle:
    score_rows = list(csv.DictReader(handle, delimiter="\t"))
base_fields = ["genotype_coverage_25", "epitope_conservation_20", "hla_coverage_15", "developability_15", "mrna_dna_manufacturability_15", "interpretability_10"]
for row in score_rows:
    base_total = sum(int(row[field]) for field in base_fields)
    assert base_total == int(row["base_total"]), row
    assert min(100, base_total + int(row["multivalency_bonus_15"])) == int(row["total_100"]), row

protein = read_fasta_sequence(OUTPUT_ROOT / "constructs" / "final-candidate.protein.fasta")
dna = read_fasta_sequence(OUTPUT_ROOT / "constructs" / "final-candidate.dna.fasta")
assert len(protein) > 300, len(protein)
assert len(dna) == (len(protein) * 3) + 9, (len(protein), len(dna))

print("All reviewer validation checks passed")
print(f"Conservation rows: {len(conservation_rows)}")
print(f"Scorecard designs: {len(score_rows)}")
print(f"Protein length: {len(protein)} aa")
print(f"DNA length: {len(dna)} nt")

## 5. Display Regenerated Figures

In [ ]:
from IPython.display import SVG, display

for figure in [
    OUTPUT_ROOT / "figures" / "e2-conservation.svg",
    OUTPUT_ROOT / "figures" / "conservation-plot.svg",
    OUTPUT_ROOT / "figures" / "scorecard-bars.svg",
    OUTPUT_ROOT / "figures" / "construct-architecture.svg",
]:
    print(figure.relative_to(PACKAGE_ROOT))
    display(SVG(filename=str(figure)))

## 6. Reviewer Checklist

- The notebook used bundled data from `final-submission/data/`.
- Regenerated files were written only to `notebook-output/`.
- No network calls, external APIs, GPUs, or wet-lab steps were required.
- Conservation scores are bounded from 0 to 1.
- Scorecard base totals and bonus-adjusted totals are coherent.
- Protein and DNA FASTA lengths are plausible for the exported antigen-only construct.